# Create Lloyd's Register Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

Lloyd's Register Foundation (LRF) is a UK charity — the grant-giving arm of Lloyd's Register — funding engineering, safety and applied-science research worldwide. This ingest covers the foundation's published grant record, one row per grant.

**Method 1 (direct open-data download).** LRF is a [360Giving](https://www.threesixtygiving.org/) publisher (registry prefix `360G-LloydsRegisterFdn`). The scraper (`scripts/local/lloyds_register_to_s3.py`) downloads the full grant corpus directly from the foundation's own site as an Excel workbook in the 360Giving standard. A reusable, column-variant-tolerant resolver handles the per-publisher header quirks. No browser automation.

**Awarding body:** Lloyd's Register Foundation - F4320310833 (GB, ROR 057q4mw47, DOI 10.13039/100008885)

**Source:** the 360Giving grants sheet. This is an **org-level grant funder** — each grant is made to a recipient **organization** (no named principal investigator). Each grant carries a stable `Identifier`, a `Title`, a `Description`, an `Amount Awarded` (GBP), real `Planned Dates:Start/End Date`, and the `Recipient Org:Name` / `Country`.

**Schema choices:**
- One row per grant. `funder_award_id` = the 360Giving `Identifier`, e.g. `360G-LloydsRegisterFdn-G0008` (stable, unique, source-authoritative).
- `display_name` = the grant's own `Title` (100% present).
- `funding_type` = `grant` uniformly. `funder_scheme` = NULL (the LRF 360Giving file has no programme column).
- **Amount** = `Amount Awarded` in GBP → `amount` (DOUBLE) + `currency` = `GBP`. LRF publishes a positive amount on 100% of rows; **§6.7 NOT waived** — populated where posted, never imputed; any `0`/blank → NULL.
- `start_date` = the real `Planned Dates:Start Date` (a true `YYYY-MM-DD` date, falling back to `Award Date`); `end_date` = the real `Planned Dates:End Date`. `start_year` / `end_year` derived from them (no false precision — these are real day-level dates).
- `lead_investigator` = an **org-only** lead: `given_name`/`family_name` NULL and `affiliation.name` = the recipient organization. `affiliation.country` = the source's `Recipient Org:Country` normalized to ISO (LRF funds worldwide — GB, US, AU, GR, NL, NG, ... ) — **source-authoritative**; values that are not a recognizable country (e.g. a bare city name) are left NULL, never guessed.
- `co_lead_investigator` and `investigators` are NULL (the source names no individuals).
- `description` = the grant's `Description`.

**S3 location:** `s3a://openalex-ingest/awards/lloyds_register/lloyds_register_grants.parquet`

**Provenance:** `lloyds_register_foundation` (verified count=0 on production 2026-06-01).


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lloyds_register_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lloyds_register/lloyds_register_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.lloyds_register_raw;


## Step 1.5: Inspect raw + country/year/coverage

Expected: 520 grants, 2008-2026. title / funder_award_id / amount / start_date / end_date / start_year / recipient_org / description 100%; recipient_country_iso ~97.5% (a few free-text values that are not a recognizable country are left NULL, never guessed).


In [ ]:
%sql
DESCRIBE openalex.awards.lloyds_register_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.lloyds_register_raw LIMIT 5;


In [ ]:
%sql
-- Recipient country distribution (source-authoritative, LRF funds worldwide).
SELECT recipient_country_iso, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year
FROM openalex.awards.lloyds_register_raw
GROUP BY recipient_country_iso
ORDER BY rows DESC;


In [ ]:
%sql
-- Top recipient organizations (sanity check parsing didn't bleed fields).
SELECT recipient_org, recipient_country_iso, COUNT(*) AS rows
FROM openalex.awards.lloyds_register_raw
WHERE recipient_org IS NOT NULL
GROUP BY recipient_org, recipient_country_iso
ORDER BY rows DESC
LIMIT 15;


## Step 1.6: Fail-fast - verify Lloyd's Register Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320310833;


## Step 2: Transform to award schema

`display_name` = the grant's own `Title`. `start_date` / `end_date` = the real Planned Start/End dates. `lead_investigator` is an org-only lead: `given_name`/`family_name` NULL, `affiliation.name` = recipient org, `affiliation.country` = the source's recipient country normalized to ISO (source-authoritative). `funder_scheme` NULL (no programme column). `co_lead_investigator` / `investigators` are NULL.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lloyds_register_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320310833  -- Lloyd's Register Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    COALESCE(n.title, CONCAT('Lloyd''s Register Foundation grant ', n.funder_award_id)) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN 'GBP' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    CAST(NULL AS STRING) AS funder_scheme,  -- LRF 360Giving file has no programme column
    'lloyds_register_foundation' AS provenance,
    TRY_CAST(n.start_date AS DATE) AS start_date,
    TRY_CAST(n.end_date AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    TRY_CAST(n.end_year AS INT) AS end_year,
    CASE
        WHEN n.recipient_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.recipient_org AS name,
                n.recipient_country_iso AS country,  -- source-authoritative (Recipient Org:Country); never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CAST(NULL AS STRING) AS landing_page_url,  -- 360Giving has no per-grant page
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.lloyds_register_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 157


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lloyds_register_foundation' AND priority = 157;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    157 AS priority  -- Lloyd's Register Foundation grants
FROM openalex.awards.lloyds_register_awards;


## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived): LRF publishes a positive amount on 100% of grants, so `pct_amount` ≈ 100%. Other checks: display_name / description / start_date / end_date / start_year ~100%; currency = [GBP]; lead_investigator.affiliation.country source-authoritative (GB majority, plus US/AU/GR/NL/...).


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.lloyds_register_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.lloyds_register_awards;


In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.lloyds_register_awards;


In [ ]:
%sql
-- §6.7 amount check (NOT waived). Expect currency = [GBP].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.lloyds_register_awards;


In [ ]:
%sql
-- Country split: lead_investigator.affiliation.country (source-authoritative).
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows,
       ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.lloyds_register_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY rows DESC;


In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.lloyds_register_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 55) AS title,
       funding_type, start_date, end_date, start_year, end_year, amount, currency,
       lead_investigator.affiliation.name AS org,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.lloyds_register_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;
